In [ ]:
# Run first: locate the repository root from any notebook subfolder.
from pathlib import Path
import os
import sys
project_root = next((candidate for candidate in [Path.cwd(), *Path.cwd().parents]
                     if (candidate / "app.py").is_file()
                     and (candidate / "retrieval_final.py").is_file()), None)
if project_root is None:
    raise RuntimeError("Start Jupyter within the project repository.")
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


In [ ]:
# Load the sampled Enron emails and evaluation questions used for chunking experiments.
import pandas as pd

sampled_emails = pd.read_csv("enron_stratified_sample.csv")
evaluation_questions = pd.read_csv("data/enron_evaluation_250.csv")

print(len(sampled_emails))
print(len(evaluation_questions))

In [ ]:
# Load the E5 embedding model and define the required query and passage prefixes.
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/e5-base-v2"
QUERY_PREFIX = "query: "
PASSAGE_PREFIX = "passage: "
embedding_model = SentenceTransformer(MODEL_NAME)

In [ ]:
# Build the searchable email text and standardise email IDs before chunking.
def build_search_text(email_frame):
    """Build the same canonical retrieval text used by the hybrid evaluation."""
    return (
        email_frame["sender"].fillna("") + " " +
        email_frame["recipients"].fillna("") + " " +
        email_frame["subject"].fillna("") + " " +
        email_frame["body"].fillna("")
    )

sampled_emails["email_id"] = sampled_emails["email_id"].astype(str)
evaluation_questions["email_id"] = evaluation_questions["email_id"].astype(str)
sampled_emails["search_text"] = build_search_text(sampled_emails)

## Resumable E5 grid search: chunk size and overlap

This experiment evaluates every combination of chunk sizes **256, 320, 400, 448** and overlaps **48, 64, 80, 96** using `intfloat/e5-base-v2`.

After each combination finishes, its metrics are saved to `e5_chunking_grid_results_250_questions.csv`.

In [ ]:
# Prepare the chunking grid, tokenize emails once, and encode the evaluation questions.
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display

CHUNK_SIZES = [256, 320, 400, 448]
CHUNK_OVERLAPS = [48, 64, 80, 96]
BATCH_SIZE = 64
CHECKPOINT_FILE = Path("e5_chunking_grid_results_250_questions.csv")

tokenizer = embedding_model.tokenizer

# Tokenize each email once and reuse the token IDs for all configurations.
tokenized_email_records = []
for _, row in sampled_emails.iterrows():
    tokenized_email_records.append({
        "email_id": row["email_id"],
        "sender": row["sender"],
        "subject": row["subject"],
        "token_ids": tokenizer.encode(
            str(row["search_text"]),
            add_special_tokens=False,
            truncation=False,
        ),
    })

# E5 requires the query prefix for question embeddings.
question_embeddings = embedding_model.encode(
    [QUERY_PREFIX + str(question) for question in evaluation_questions["question"]],
    batch_size=BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

print(
    f"Prepared {len(tokenized_email_records):,} emails and "
    f"{len(evaluation_questions):,} evaluation questions."
)

In [ ]:
# Define functions to create chunks and evaluate each chunking configuration using retrieval recall.

# Split sampled emails into overlapping E5-token chunks for one parameter combination.
def create_chunks(chunk_size, chunk_overlap):
    """Create E5-token chunks for one parameter combination."""
    if chunk_overlap >= chunk_size:
        raise ValueError("chunk_overlap must be smaller than chunk_size")
    if chunk_size >= embedding_model.max_seq_length:
        raise ValueError("chunk_size must remain below E5's maximum sequence length")

    records = []
    step = chunk_size - chunk_overlap

    for email_record in tokenized_email_records:
        token_ids = email_record["token_ids"]
        for chunk_number, start in enumerate(range(0, len(token_ids), step)):
            end = min(start + chunk_size, len(token_ids))
            chunk_token_ids = token_ids[start:end]
            if not chunk_token_ids:
                continue
            chunk_text = tokenizer.decode(chunk_token_ids, skip_special_tokens=True)
            records.append({
                "email_id": email_record["email_id"],
                "chunk_number": chunk_number,
                "chunk_id": f'{email_record["email_id"]}_chunk_{chunk_number}',
                "chunk_text": chunk_text,
                "embedding_text": PASSAGE_PREFIX + chunk_text,
                "sender": email_record["sender"],
                "subject": email_record["subject"],
            })
            if end == len(token_ids):
                break

    return pd.DataFrame(records)

# Build and evaluate one chunking configuration against the labelled source emails.
def evaluate_configuration(chunk_size, chunk_overlap, k=10):
    """Build, embed, retrieve, and score one E5 chunking configuration."""
    chunks = create_chunks(chunk_size, chunk_overlap)
    chunk_embeddings = embedding_model.encode(
        chunks["embedding_text"].tolist(),
        batch_size=BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    # Normalized vectors make dot product equivalent to cosine similarity.
    similarities = question_embeddings @ chunk_embeddings.T
    chunk_email_ids = chunks["email_id"].to_numpy()
    relevant_email_ids = evaluation_questions["email_id"].astype(str).to_numpy()

    hits = {1: 0, 5: 0, 10: 0}
    for question_index, scores in enumerate(similarities):
        # Sort chunks, then retain the first and best scoring chunk per email.
        ranked_chunk_indices = np.argsort(scores)[::-1]
        retrieved_email_ids = []
        seen = set()
        for chunk_index in ranked_chunk_indices:
            email_id = str(chunk_email_ids[chunk_index])
            if email_id not in seen:
                seen.add(email_id)
                retrieved_email_ids.append(email_id)
                if len(retrieved_email_ids) == k:
                    break

        relevant_id = relevant_email_ids[question_index]
        for cutoff in hits:
            hits[cutoff] += int(relevant_id in retrieved_email_ids[:cutoff])

    return {
        "chunk_size": chunk_size,
        "chunk_overlap": chunk_overlap,
        "number_of_chunks": len(chunks),
        "recall_at_1": hits[1] / len(evaluation_questions),
        "recall_at_5": hits[5] / len(evaluation_questions),
        "recall_at_10": hits[10] / len(evaluation_questions),
    }

In [ ]:
# Run the resumable chunking grid search and rank configurations by retrieval performance.

RESULT_COLUMNS = [
    "chunk_size", "chunk_overlap", "number_of_chunks",
    "recall_at_1", "recall_at_5", "recall_at_10",
]

# Resume from the E5 checkpoint if one already exists.
if CHECKPOINT_FILE.exists():
    grid_results = pd.read_csv(CHECKPOINT_FILE)
    print(f"Loaded {len(grid_results)} completed combination(s) from {CHECKPOINT_FILE}.")
else:
    grid_results = pd.DataFrame(columns=RESULT_COLUMNS)
    print("No E5 checkpoint found; starting a new grid search.")

completed = {
    (int(row.chunk_size), int(row.chunk_overlap))
    for row in grid_results.itertuples()
}
all_combinations = [
    (chunk_size, chunk_overlap)
    for chunk_size in CHUNK_SIZES
    for chunk_overlap in CHUNK_OVERLAPS
]
total_combinations = len(all_combinations)

for run_number, (chunk_size, chunk_overlap) in enumerate(all_combinations, start=1):
    combination = (chunk_size, chunk_overlap)
    if combination in completed:
        print(
            f"[{run_number}/{total_combinations}] Skipping completed: "
            f"size={chunk_size}, overlap={chunk_overlap}"
        )
        continue

    remaining = total_combinations - len(completed)
    print(
        f"[{run_number}/{total_combinations}] Running: size={chunk_size}, "
        f"overlap={chunk_overlap} ({remaining} combination(s) still to complete)"
    )
    result = evaluate_configuration(chunk_size, chunk_overlap)
    grid_results = pd.concat(
        [grid_results, pd.DataFrame([result])],
        ignore_index=True,
    )

    # Save immediately so completed E5 runs survive an interruption.
    grid_results[RESULT_COLUMNS].to_csv(CHECKPOINT_FILE, index=False)
    completed.add(combination)
    print(
        f"Saved result to {CHECKPOINT_FILE} "
        f"({len(completed)}/{total_combinations} complete)."
    )

grid_results = grid_results[RESULT_COLUMNS].copy()
grid_results["mean_recall"] = grid_results[
    ["recall_at_1", "recall_at_5", "recall_at_10"]
].mean(axis=1)
grid_results = grid_results.sort_values(
    ["mean_recall", "recall_at_10", "recall_at_5"],
    ascending=False,
).reset_index(drop=True)

print(
    f"E5 grid-search results available: "
    f"{len(grid_results)}/{total_combinations}"
)
display(
    grid_results.style
    .format({
        "recall_at_1": "{:.1%}",
        "recall_at_5": "{:.1%}",
        "recall_at_10": "{:.1%}",
        "mean_recall": "{:.1%}",
        "number_of_chunks": "{:,}",
    })
    .background_gradient(
        subset=["recall_at_1", "recall_at_5", "recall_at_10", "mean_recall"],
        cmap="YlGn",
    )
)

In [ ]:
# Report the highest-performing E5 chunk size and overlap combination.
best = grid_results.iloc[0]
print(
    f'Best E5 configuration: chunk size {int(best["chunk_size"])}, '
    f'overlap {int(best["chunk_overlap"])}\n'
    f'Recall@1: {best["recall_at_1"]:.1%} | '
    f'Recall@5: {best["recall_at_5"]:.1%} | '
    f'Recall@10: {best["recall_at_10"]:.1%} | '
    f'Mean recall: {best["mean_recall"]:.1%}'
)

In [ ]:
# Visualise how chunk size and overlap affect E5 retrieval recall.
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), constrained_layout=True)

for ax, metric, title in zip(
    axes,
    ["recall_at_1", "recall_at_5", "recall_at_10"],
    ["Recall@1", "Recall@5", "Recall@10"],
):
    heatmap_data = grid_results.pivot(
        index="chunk_size",
        columns="chunk_overlap",
        values=metric,
    ).sort_index()
    sns.heatmap(
        heatmap_data,
        annot=True,
        fmt=".1%",
        cmap="YlGn",
        vmin=0,
        vmax=1,
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel("Chunk overlap")
    ax.set_ylabel("Chunk size")

fig.suptitle("E5 chunking grid-search results", fontsize=15)
plt.show()